Packages:

In [1]:
import pandas as pd
import numpy as np
import wrds

Data sourcing:

In [2]:
db = wrds.Connection(wrsd_username="lucadesj")

Enter your WRDS username [lucad]: lucadesj
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [3]:
# Extract fundamentals annual
# Table : comp.funda

compustat_query = """
    SELECT
        gvkey, datadate, fyear, indfmt,
        conm,
        -- Compte de résultat
        sale, ebit, oibdp, dp, ni, txt, xint, oancf, capx, cogs, revt,
        -- Bilan
        at, ceq, dltt, dlc, che, lct,
        -- Actions & prix
        csho, prcc_f,
        -- Intangible
        xrd, xsga
    FROM comp.funda
    WHERE fyear BETWEEN 2000 AND 2025
      AND indfmt  = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'D'
      AND consol  = 'C'
      AND sale > 0
      AND at   > 0
      AND ceq  > 0
"""


db_comp = db.raw_sql(compustat_query, date_cols=['datadate'])
db_comp = db_comp.drop_duplicates(["gvkey", "fyear"])
db_comp.head()

,gvkey,datadate,fyear,indfmt,conm,sale,ebit,oibdp,dp,ni,...,at,ceq,dltt,dlc,che,lct,csho,prcc_f,xrd,xsga
0,001004,2001-05-31,2000,INDL,AAR CORP,874.255,45.79,64.367,18.577,18.531,...,701.854,340.212,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
1,001010,2000-12-31,2000,INDL,ACF INDUSTRIES INC,443.8,117.0,178.7,61.7,79.9,...,3794.5,985.2,1866.8,176.1,113.0,<NA>,0.015,<NA>,<NA>,32.4
2,001013,2000-10-31,2000,INDL,ADC TELECOMMUNICATIONS INC,3287.9,523.9,670.1,146.2,868.1,...,3970.5,2912.7,16.5,28.5,1354.2,1041.3,770.3,21.375,338.0,1050.7
3,001019,2000-12-31,2000,INDL,AFA PROTECTIVE SYSTEMS INC,42.504,3.415,5.443,2.028,1.762,...,28.638,13.184,1.12,0.666,1.818,5.39,0.166,183.0,<NA>,12.554
4,001021,2000-06-30,2000,INDL,AFP IMAGING CORP,25.367,-0.028,0.672,0.7,-0.808,...,11.608,4.455,4.875,0.195,0.434,2.278,9.271,0.49,0.477,8.189


In [4]:
# Extract Monthly Price
# Table : crsp.msf 

crsp_query = """
   SELECT
    m.permno,
    m.date,
    m.prc,
    m.ret,
    m.retx,   
    m.shrout,
    n.shrcd,
    n.exchcd
FROM crsp.msf AS m
JOIN crsp.msenames AS n
    ON m.permno = n.permno
    AND m.date BETWEEN n.namedt AND n.nameendt
WHERE n.shrcd IN (10, 11)        -- Actions ordinaires uniquement
    AND n.exchcd IN (1, 2, 3)    -- NYSE, AMEX, et NASDAQ
    AND m.date >= '2000-01-01'
"""
db_crsp = db.raw_sql(crsp_query, date_cols=['date'])
db_crsp['date'] = db_crsp['date'] + pd.offsets.MonthEnd(0) #Ramène chaque date à la fin du mois
db_crsp.head()

,permno,date,prc,ret,retx,shrout,shrcd,exchcd
0,10032,2000-01-31,46.5,0.056818,0.056818,17638.0,11,3
1,10032,2000-02-29,56.46875,0.214382,0.214382,17638.0,11,3
2,10032,2000-03-31,66.625,0.179856,0.179856,17810.0,11,3
3,10032,2000-04-30,76.625,0.150094,0.150094,17810.0,11,3
4,10032,2000-05-31,83.5,0.089723,0.089723,18259.0,11,3


In [5]:
test_query = """ 
    SELECT 
        permno, 
        dlstdt, 
        dlret, 
        dlstcd, 
        dlprc 
    FROM crsp.msedelist
    WHERE dlstdt >= '2000-01-01'
"""

db_dl = db.raw_sql(test_query, date_cols=['dlstdt'])
db_dl['date'] = db_dl['dlstdt'] + pd.offsets.MonthEnd(0)
db_dl.head()

,permno,dlstdt,dlret,dlstcd,dlprc,date
0,10001,2017-08-03,0.011583,233,0.0,2017-08-31
1,10002,2013-02-15,0.046007,231,0.0,2013-02-28
2,10009,2000-11-03,0.005671,233,0.0,2000-11-30
3,10012,2005-08-02,-0.058824,552,0.16,2005-08-31
4,10016,2001-05-03,-0.028936,231,0.0,2001-05-31


In [6]:
db_crsp = db_crsp.merge(
    db_dl[['permno', 'date', 'dlret', 'dlstcd', 'dlprc']],
    on=['permno', 'date'],
    how='left')

In [7]:
# Trouver le dernier mois de chaque permno dans le CRSP
last_obs = db_crsp.groupby('permno')['date'].max().reset_index(name='last_date')

orphan_dl = db_dl.merge(last_obs, on='permno')
orphan_dl = orphan_dl[orphan_dl['date'] > orphan_dl['last_date']].copy()

# Calculer le gap en mois entre dlstdt et le dernier mois dans msf
orphan_dl['gap_months'] = (
    (orphan_dl['date'].dt.year - orphan_dl['last_date'].dt.year) * 12
    + (orphan_dl['date'].dt.month - orphan_dl['last_date'].dt.month)
)

orphan_valid = orphan_dl[orphan_dl['gap_months'] <= 6]  

if not orphan_valid.empty:
    db_crsp = db_crsp.merge(
        orphan_valid[['permno', 'last_date', 'dlret', 'dlstcd', 'dlprc']]
              .rename(columns={'last_date': 'date',
                               'dlret': 'dlret_orphan',
                               'dlstcd': 'dlstcd_orphan',
                               'dlprc': 'dlprc_orphan'}),
        on=['permno', 'date'],
        how='left'
    )
    db_crsp['dlret']  = db_crsp['dlret'].fillna(db_crsp['dlret_orphan'])
    db_crsp['dlstcd'] = db_crsp['dlstcd'].fillna(db_crsp['dlstcd_orphan'])
    db_crsp['dlprc']  = db_crsp['dlprc'].fillna(db_crsp['dlprc_orphan'])
    db_crsp.drop(columns=['dlret_orphan', 'dlstcd_orphan', 'dlprc_orphan'], inplace=True)


In [8]:
print(db_comp.shape)
print(db_crsp.shape)

(178300, 26)
(1294973, 11)


In [9]:
#Link table :
link_query = """
    SELECT gvkey, lpermno AS permno, linkdt, linkenddt
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LC', 'LU')
      AND linkprim IN ('P', 'C')
"""

db_link = db.raw_sql(link_query, date_cols=['linkdt', 'linkenddt'])
db_link['linkenddt'] = db_link['linkenddt'].fillna(pd.Timestamp('2099-12-31')) #Remplace les NaN par une date fictive future (lien encore actif)

db_link.head()

,gvkey,permno,linkdt,linkenddt
0,001000,25881.0,1970-11-13,1978-06-30
1,001001,10015.0,1983-09-20,1986-07-31
2,001002,10023.0,1972-12-14,1973-06-05
3,001003,10031.0,1983-12-07,1989-08-16
4,001004,54594.0,1972-04-24,2099-12-31


In [10]:
#Panel final

#Prise en compte du décalage entre les données et la date de clôture de l'année fiscale 
db_comp['avail_date'] = (db_comp['datadate'] + pd.DateOffset(months=6)) + pd.offsets.MonthEnd(0)

#Conservation des dates valides uniquement
crsp_linked = db_crsp.merge(db_link, on='permno', how='inner')

crsp_linked = crsp_linked[
    (crsp_linked['date'] >= crsp_linked['linkdt']) &
    (crsp_linked['date'] <= crsp_linked['linkenddt'])].drop(columns=['linkdt', 'linkenddt']) #Ne garde uniquement les valeurs comprises dans l'interval de validité du lien des "keys"


#Liaison entre chaque mois du CRSP et le dernier exercice Compustat dispo 

crsp_linked['date'] = pd.to_datetime(crsp_linked['date'])
db_comp['avail_date'] = pd.to_datetime(db_comp['avail_date'])

crsp_linked = crsp_linked.sort_values(['gvkey', 'date'])
db_comp = db_comp.sort_values(['gvkey', 'avail_date'])

In [11]:
import psutil
print(f"RAM utilisée : {psutil.virtual_memory().percent}%")
print(f"Swap utilisé : {psutil.swap_memory().percent}%")

RAM utilisée : 28.9%
Swap utilisé : 0.0%


In [12]:
# Merge sur gvkey 
temp = crsp_linked.merge(
    db_comp[['gvkey', 'avail_date', 'fyear', 'conm', 'indfmt', 'ni', 'ceq', 'oancf', 'sale', 
             'ebit', 'oibdp', 'dp', 'txt', 'xint', 'capx', 'revt','cogs', 'at', 'dltt', 
             'dlc', 'che', 'lct', 'csho', 'prcc_f', 'xrd', 'xsga']],
    on='gvkey', 
    how='left', 
    suffixes=('', '_comp')
)

# On garde uniquement les lignes où date CRSP <= date dispo comptes
temp = temp[temp['date'] <= temp['avail_date']]

# Pour chaque (gvkey, date CRSP), on prend le compte le plus récent disponible
panel = (temp
    .sort_values(['gvkey', 'date', 'avail_date'])
    .groupby(['gvkey', 'date'])
    .first() 
    .reset_index()
)

# Nettoyage des doublons colonnes
panel = panel.drop(columns=['avail_date_comp'], errors='ignore')

print(f"Panel : {panel.shape[0]:,}  obs — {panel['permno'].nunique():,} titres")


Panel : 1,189,596  obs — 10,823 titres


In [13]:
panel.head()

,gvkey,date,permno,prc,ret,retx,shrout,shrcd,exchcd,dlret,...,cogs,at,dltt,dlc,che,lct,csho,prcc_f,xrd,xsga
0,001004,2000-01-31,54594,17.6875,-0.009199,-0.013937,27181.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
1,001004,2000-02-29,54594,23.75,0.342756,0.342756,27181.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
2,001004,2000-03-31,54594,16.6875,-0.297368,-0.297368,27181.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
3,001004,2000-04-30,54594,15.0625,-0.092285,-0.097378,26963.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
4,001004,2000-05-31,54594,13.875,-0.078838,-0.078838,26963.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077


Traitement des NaN:

In [14]:
#Remplacement par 0 
zero_fill = ['xrd', 'capx', 'dp', 'txt', 'xint']
panel[zero_fill] = panel[zero_fill].fillna(0)

In [15]:
#Remplacement par lissage vers l'avant (ou l'arrière si pas dispo)
balance_sheet = ['dltt', 'dlc']
for col in balance_sheet:
    panel[col] = (panel
                  .groupby('gvkey')[col]
                  .ffill()
                  .bfill()
                  .fillna(0))

In [16]:
#Traitement des valeurs manquantes lct (current liabilities) 
print(db_comp['indfmt'].value_counts()) #Contrôle pour vérifier s'il reste des banques

lct_missing = panel.groupby('gvkey')['lct'].apply(lambda x: x.isna().mean())
print("LCT manquants par gvkey :")
print(lct_missing.describe()) #moyenne de manquement correct

bad_lct = lct_missing[lct_missing > 0.8].index.tolist()
print(f"Entreprises LCT >80% manquant : {len(bad_lct)}") #Résultat : 1806, résultat attendu (correspond aux PME sans passif)

panel['lct'] = (panel
                .groupby('gvkey')['lct']
                .ffill()
                .bfill()
                .fillna(0))

indfmt
INDL    178300
Name: count, dtype: Int64
LCT manquants par gvkey :
count    10707.000000
mean         0.170321
std          0.374828
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: lct, dtype: float64
Entreprises LCT >80% manquant : 1806


In [17]:
#Traitement des valeurs manquantes oibdp (Operatinf Income before Depreciation

oibdp_missing = panel.groupby('gvkey')['oibdp'].apply(lambda x: x.isna().mean())
oibdp_missing.describe()

bad_oibdp = oibdp_missing[oibdp_missing == 1].index.tolist() 
print(len(bad_oibdp))

bad_companies = panel[panel['gvkey'].isin(bad_oibdp)][['gvkey', 'conm']].drop_duplicates()
print(bad_companies) 

# 25 entreprises sur des secteurs diversifiés, on fait le choix de les exclures de notre sample

panel = panel[~panel['gvkey'].isin(bad_oibdp)]

panel['oibdp'] = (panel
                .groupby('gvkey')['oibdp']
                .ffill()
                .bfill())

25
          gvkey                          conm
3288     001115             ADAC LABORATORIES
11601    001465         AMERICAN GENERAL CORP
52180    002965           CHEMFAB CORPORATION
60659    003230       COMDISCO HOLDING CO INC
329288   013532           MERCHANTS GROUP INC
394201   017243            TRENWICK GROUP LTD
416259   018792        GAMING & LEISURE PPTYS
426397   019440           LADDER CAPITAL CORP
528054   025573             AXA FINANCIAL INC
540962   026020           ANTEX BIOLOGICS INC
546163   026410  FOUR CORNERS PROPERTY TR INC
548407   026839      PHILLIPS EDISON & CO INC
551614   027394         PARK HOTELS & RESORTS
615457   030165   ASSURED GUARANTY MUNI HLDGS
671116   035305                  VERICITY INC
689850   038753            ENACT HOLDINGS INC
699778   040092           ARCHER AVIATION INC
714144   061302        ARCH CAPITAL GROUP LTD
818858   065175    CORSAIR COMMUNICATIONS INC
899353   117701                  ALLAIRE CORP
925646   122435                

In [18]:
#Traitement des valeurs manquantes xsga (Selling, General and Administrative Expenses)

# xsga = Revenue - COGS - Depreciation - Op. Income

panel['xsga'] = panel['xsga'].fillna(panel['revt'] - panel['cogs'] - panel['oibdp'])


In [19]:
#Traitement des valeurs manquantes oancf (Operating Activity - Net Cash Flow)

oancf_missing = panel.groupby('gvkey')['oancf'].apply(lambda x: x.isna().mean())
oancf_missing.describe()

bad_oancf = oancf_missing[oancf_missing == 1].index.tolist() 
print(len(bad_oancf))

bad_companies1 = panel[panel['gvkey'].isin(bad_oancf)][['gvkey', 'conm']].drop_duplicates() #on s'aperçoit que le filtre INDL au moment de la query n'a pas bien fonctionné car la 
                                                                                            #plupart des entreprises sont ici des banques
print(bad_companies1) 

#Par soucis de simplicité, et comme ce sont majoritairement des banques, on drop 

panel = panel[~panel['gvkey'].isin(bad_oancf)]

#Le peu de valeurs manquantes restant (407) sont lissées

panel['oancf'] = (panel
                .groupby('gvkey')['oancf']
                .ffill()
                .bfill())


233
          gvkey                        conm
27477    001998               BANK ONE CORP
28131    002007        FIRST BANKS AMER INC
101069   004682          U S BANCORP/DE-OLD
101982   004708               BANCWEST CORP
102737   004742    FIRST VIRGINIA BANKS INC
...         ...                         ...
962449   133988         PORT FINANCIAL CORP
971042   137378    FIRST SHARES BANCORP INC
971581   137531    DUTCHFORK BANCSHARES INC
974931   138128          PACIFIC UNION BANK
1039914  157973  FIRST NATL BANKSHRS FL INC

[233 rows x 2 columns]


In [20]:
#Traitement des valeurs manquantes ebit 

# EBIT = OIBDP - DP

panel['ebit'] = panel['ebit'].fillna(panel['oibdp'] - panel['dp'])

In [21]:
#Traitement des valeurs manquantes csho (Common Share Outstanding)

# csho (compustat) peut être rapproché à shrout (CRSP), seulement dans la documentation respective on voit que l'unité est traité différemment

# csho = shrout / 1000 

panel['csho'] = panel['csho'].fillna(panel['shrout'] / 1000) 

#l'utilité de garder les deux peuvent être discuté. Cependant, en réalité elles sont différentes d'un point de vue comptable (ici on a simplifié pour le peu de valeur manquantes (400)

In [46]:
#Traitement des prc manquants
panel = panel.sort_values(['permno', 'date']).reset_index(drop=True)

# Groupes de NaN consécutifs par permno
panel['_prc_null'] = panel['prc'].isna().astype(int)

# Longueur max de la séquence de NaN consécutifs pour chaque permno
panel['_gap_id'] = (panel.groupby('permno')['_prc_null'].transform(lambda x: (x != x.shift()).cumsum()))

panel['_gap_size'] = (panel.groupby(['permno', '_gap_id'])['_prc_null'].transform('sum'))

# Appliquer ffill uniquement aux permnos dont le gap max est < 4
permno_ffill = set(panel[panel['_prc_null'] == 1]
    .groupby('permno')['_gap_size']
    .max()
    .loc[lambda x: x < 3]
    .index)

filtre_ffill = panel['prc'].isna() & panel['permno'].isin(permno_ffill)

panel.loc[panel['permno'].isin(permno_ffill), 'prc'] = (panel.loc[panel['permno'].isin(permno_ffill)].groupby('permno')['prc'].ffill().values)

panel.drop(columns=['_prc_null', '_gap_id', '_gap_size'], inplace=True)

# Remplacer prc manquant par dlprc s'il est disponible 

filtre_dlprc = panel['prc'].isna() & panel['dlprc'].notna()

panel.loc[filtre_dlprc, 'prc'] = panel.loc[filtre_dlprc, 'dlprc']


# Traitement des prcc_f (prix à la clôture de l'année fiscale)

filtre_prccf = panel['prcc_f'].isna() & panel['prc'].notna()

panel.loc[filtre_prccf, 'prcc_f'] = panel.loc[filtre_prccf, 'prc']

In [44]:
#Traitement des valeurs manquantes des returns et prise en compte du Survivorship Bias

panel = panel.sort_values(['permno', 'date']).reset_index(drop=True)

#Suppression des premières observations (ret incalculable sans prc_lag)
panel['first_date'] = panel.groupby('permno')['date'].transform('min')

filtre_drop = (
    (panel['date'] == panel['first_date']) &
    panel['ret'].isna())

panel = panel[~filtre_drop].reset_index(drop=True)
panel.drop(columns=['first_date'], inplace=True)


#Traitement des trous au milieu des séries

panel['_prc']     = panel['prc'].abs()
panel['_prc_lag'] = panel.groupby('permno')['_prc'].shift(1)

# Identifier première et dernière observation de chaque permno
panel['first_date'] = panel.groupby('permno')['date'].transform('min')
panel['last_date']  = panel.groupby('permno')['date'].transform('max')

filtre_trous = (
    panel['ret'].isna() &
    (panel['date'] != panel['first_date']) &
    (panel['date'] != panel['last_date']) &
    panel['_prc'].notna() &
    panel['_prc_lag'].notna() &
    (panel['_prc_lag'] != 0))

panel.loc[filtre_trous, 'ret'] = (panel.loc[filtre_trous, '_prc'] / panel.loc[filtre_trous, '_prc_lag'] - 1)

panel.drop(columns=['_prc', '_prc_lag', 'first_date', 'last_date'], inplace=True)



# Remplacer ret manquant par dlret s'il est disponible 

filtre_dlret = panel['ret'].isna() & panel['dlret'].notna()

panel.loc[filtre_dlret, 'ret'] = panel.loc[filtre_dlret, 'dlret']


# Identifier les permnos ayant encore des prc manquants

permno_prc_manquant = set(panel[panel['prc'].isna()]['permno'].unique())
panel = panel[~panel['permno'].isin(permno_prc_manquant)].reset_index(drop=True)



#Supprimer les erreurs de la base de données 

panel['last_date'] = panel.groupby('permno')['date'].transform('max')

filtre_last_na = (panel['date'] == panel['last_date']) & panel['ret'].isna()

panel = panel[~filtre_last_na].reset_index(drop=True)
panel.drop(columns=['last_date'], inplace=True)


#Traitement des retx
filtre_retx = panel['retx'].isna() & panel['ret'].notna()

panel.loc[filtre_retx, 'retx'] = panel.loc[filtre_retx, 'ret']


In [48]:
#Suppression des colonnes inutiles

panel.drop(columns=['dlret', 'dlstcd', 'dlprc'], inplace=True)

In [50]:
panel.isna().sum()

gvkey         0
date          0
permno        0
prc           0
ret           0
retx          0
shrout        0
shrcd         0
exchcd        0
avail_date    0
fyear         0
conm          0
indfmt        0
ni            0
ceq           0
oancf         0
sale          0
ebit          0
oibdp         0
dp            0
txt           0
xint          0
capx          0
revt          0
cogs          0
at            0
dltt          0
dlc           0
che           0
lct           0
csho          0
prcc_f        0
xrd           0
xsga          0
dtype: int64

In [49]:
#Visualisation de la DB

panel

,gvkey,date,permno,prc,ret,retx,shrout,shrcd,exchcd,avail_date,...,cogs,at,dltt,dlc,che,lct,csho,prcc_f,xrd,xsga
0,012994,2000-01-31,10001,8.125,-0.044118,-0.044118,2450.0,11,3,2000-12-31,...,67.2,50.553,16.395,5.3,0.112,14.841,2.475,8.0,0.0,-0.0
1,012994,2000-02-29,10001,8.25,0.015385,0.015385,2450.0,11,3,2000-12-31,...,67.2,50.553,16.395,5.3,0.112,14.841,2.475,8.0,0.0,-0.0
2,012994,2000-03-31,10001,-8.0,-0.015758,-0.030303,2464.0,11,3,2000-12-31,...,67.2,50.553,16.395,5.3,0.112,14.841,2.475,8.0,0.0,-0.0
3,012994,2000-04-30,10001,-8.09375,0.011719,0.011719,2464.0,11,3,2000-12-31,...,67.2,50.553,16.395,5.3,0.112,14.841,2.475,8.0,0.0,-0.0
4,012994,2000-05-31,10001,-7.90625,-0.023166,-0.023166,2464.0,11,3,2000-12-31,...,67.2,50.553,16.395,5.3,0.112,14.841,2.475,8.0,0.0,-0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1176326,184996,2024-08-31,93436,214.11,-0.077391,-0.077391,3194640.0,11,3,2025-06-30,...,74872.0,122070.0,10360.0,3263.0,37057.0,28821.0,3216.0,403.84,4540.0,9690.0
1176327,184996,2024-09-30,93436,261.63,0.221942,0.221942,3207000.0,11,3,2025-06-30,...,74872.0,122070.0,10360.0,3263.0,37057.0,28821.0,3216.0,403.84,4540.0,9690.0
1176328,184996,2024-10-31,93436,249.85001,-0.045025,-0.045025,3210060.0,11,3,2025-06-30,...,74872.0,122070.0,10360.0,3263.0,37057.0,28821.0,3216.0,403.84,4540.0,9690.0
1176329,184996,2024-11-30,93436,345.16,0.381469,0.381469,3210060.0,11,3,2025-06-30,...,74872.0,122070.0,10360.0,3263.0,37057.0,28821.0,3216.0,403.84,4540.0,9690.0
